# 03 — Personalized Recommendations

**Project:** AI-Based Personalized Learning and Student Performance Predictor (P38)

**Purpose:** Use the saved Logistic Regression model to make a pre-exam mastery prediction and provide a separate, rule-based recommendation from actual exam scores.

We use the original, unweighted Logistic Regression model because it had the highest validation macro-F1 among the models we compared. The class-weighting experiment remains documented in Notebook 02, but it did not improve validation macro-F1.

**Notebook roadmap**
- **Section 1:** Load the saved model and its fitted preprocessor.
- **Section 2:** Predict a mastery category and suggest a practice level from five categorical inputs.
- **Section 3:** Recommend a subject to review using actual exam scores.
- **Section 4:** Verify the workflow and revise key limitations and viva questions.

**Important limitation:** The dataset has no question-level difficulty labels or topic-level scores. This notebook predicts an engineered student mastery category; it does not directly predict individual question difficulty.


## Section 1 — Load Saved Artifacts

In this section, we load the trained Logistic Regression model and the fitted preprocessing object saved by Notebook 02. We do not retrain the model here.


### 1.1 Import Libraries

We import only the libraries needed to load the artifacts, validate example inputs, and display results.


In [10]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

### 1.2 Locate the Project Folder and Artifacts

The notebook can be opened from either the project root or the `notebooks/` directory. We use relative paths rather than hardcoding a machine-specific username.


In [11]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "models").is_dir():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").is_dir():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the project's models/ folder. "
        "Open this notebook from the project root or its notebooks/ folder."
    )

MODELS_DIR = PROJECT_ROOT / "models"
LOGISTIC_MODEL_PATH = MODELS_DIR / "logistic_regression.joblib"
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"

for path in [LOGISTIC_MODEL_PATH, PREPROCESSOR_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact not found: {path}\n"
            "Run Notebook 02 successfully before running this notebook."
        )

print(f"Project root: {PROJECT_ROOT}")
print("Required model artifacts were found.")

Project root: /home/waterball/Desktop/AI-Student-Performance-Predictor
Required model artifacts were found.


### 1.3 Load the Model and Preprocessor

The **preprocessor** converts categorical inputs into the same one-hot encoded representation used during training. The **Logistic Regression model** then estimates the probability of each mastery category.

We must reuse the saved preprocessor rather than fit a new one, because a newly fitted encoder could produce a different feature layout.


In [12]:
preprocessor = joblib.load(PREPROCESSOR_PATH)
logistic_model = joblib.load(LOGISTIC_MODEL_PATH)

CATEGORICAL_FEATURES = [
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course",
]
CLASS_NAMES = ["Low", "Medium", "High"]

# Read permitted category values from the fitted OneHotEncoder.
encoder = preprocessor.named_transformers_["categorical"]
allowed_categories = {
    feature: [str(value) for value in categories]
    for feature, categories in zip(CATEGORICAL_FEATURES, encoder.categories_)
}

print("Model classes:", list(logistic_model.classes_))
print("Input features:", CATEGORICAL_FEATURES)
print("\nAllowed values learned by the fitted encoder:")
for feature, values in allowed_categories.items():
    print(f"- {feature}: {values}")

Model classes: ['High', 'Low', 'Medium']
Input features: ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

Allowed values learned by the fitted encoder:
- gender: ['female', 'male']
- race/ethnicity: ['group A', 'group B', 'group C', 'group D', 'group E']
- parental level of education: ["associate's degree", "bachelor's degree", 'high school', "master's degree", 'some college', 'some high school']
- lunch: ['free/reduced', 'standard']
- test preparation course: ['completed', 'none']


## Section 2 — Pre-exam Prediction

This mode estimates a student's mastery category before exam scores are available. It accepts the same five categorical features used during training, then maps the predicted category to a broad practice suggestion.


### 2.1 Define the Practice Recommendation Rules

These recommendations are simple, explicitly defined rules. They are not generated by a separately trained recommendation model.


In [13]:
PRACTICE_RECOMMENDATIONS = {
    "Low": (
        "Start with foundational practice. Review core concepts and work through "
        "guided, easier questions before progressing."
    ),
    "Medium": (
        "Practise core concepts with a mix of easy and moderate questions. "
        "Review mistakes and gradually increase difficulty."
    ),
    "High": (
        "Practise challenging application and multi-step questions, while using "
        "mixed practice to maintain existing skills."
    ),
}

print("Recommendation rules are ready for all three mastery categories.")

Recommendation rules are ready for all three mastery categories.


### 2.2 Build a Reusable Prediction Function

The function checks that all five required features are present and that their values are known to the fitted encoder. It then transforms the inputs with the saved preprocessor and obtains a prediction and class probabilities from Logistic Regression.

The probabilities are model estimates, not guarantees or causal explanations.


In [14]:
def predict_mastery(student_profile):
    """Predict mastery from the five categorical inputs and return probabilities."""
    missing_features = [
        feature for feature in CATEGORICAL_FEATURES
        if feature not in student_profile
    ]
    extra_features = [
        feature for feature in student_profile
        if feature not in CATEGORICAL_FEATURES
    ]

    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
    if extra_features:
        raise ValueError(f"Unexpected features: {extra_features}")

    for feature in CATEGORICAL_FEATURES:
        if student_profile[feature] not in allowed_categories[feature]:
            raise ValueError(
                f"Invalid value for '{feature}': {student_profile[feature]!r}. "
                f"Choose one of: {allowed_categories[feature]}"
            )

    input_df = pd.DataFrame(
        [{feature: student_profile[feature] for feature in CATEGORICAL_FEATURES}],
        columns=CATEGORICAL_FEATURES,
    )

    encoded_input = preprocessor.transform(input_df)
    predicted_category = str(logistic_model.predict(encoded_input)[0])
    probability_values = logistic_model.predict_proba(encoded_input)[0]

    probabilities = pd.DataFrame({
        "Mastery Level": logistic_model.classes_,
        "Probability": probability_values,
    }).sort_values("Probability", ascending=False).reset_index(drop=True)

    return {
        "input": input_df,
        "predicted_mastery": predicted_category,
        "probabilities": probabilities,
        "recommendation": PRACTICE_RECOMMENDATIONS[predicted_category],
    }

### 2.3 Try a Sample Pre-exam Prediction

The following example demonstrates the expected input structure. You can change the values, but they must match the allowed categories printed in Section 1.


In [15]:
sample_student = {
    "gender": "female",
    "race/ethnicity": "group C",
    "parental level of education": "bachelor's degree",
    "lunch": "standard",
    "test preparation course": "completed",
}

sample_result = predict_mastery(sample_student)

display(sample_result["input"])
print("Predicted mastery level:", sample_result["predicted_mastery"])
print("\nClass probabilities:")
display(sample_result["probabilities"].style.format({"Probability": "{:.1%}"}))
print("\nSuggested practice:")
print(sample_result["recommendation"])

,gender,race/ethnicity,parental level of education,lunch,test preparation course
0,female,group C,bachelor's degree,standard,completed


Predicted mastery level: High

Class probabilities:


,Mastery Level,Probability
0,High,66.8%
1,Medium,32.4%
2,Low,0.8%



Suggested practice:
Practise challenging application and multi-step questions, while using mixed practice to maintain existing skills.


### 2.4 Understand the Prediction Output

- **Predicted mastery level:** the category selected by the model.
- **Class probabilities:** the model's estimated probability for each category; together they should sum to approximately 100%.
- **Suggested practice:** a recommendation selected by the explicit rule for the predicted category.

The prediction is an estimate based on this dataset and the five supplied inputs. It is not a guarantee about an individual student.


## Section 3 — Post-exam Recommendation

This mode is separate from the trained classifier. It uses actual Mathematics, Reading, and Writing scores to identify the lowest score and suggest a subject to review. The rule can also handle ties.

Because the dataset contains only subject-level scores, it cannot identify specific weak topics within a subject.


### 3.1 Define the Score-based Recommendation Rule

The function validates that each score is numeric and lies between 0 and 100, identifies the lowest score, and returns a corresponding recommendation. This is rule-based logic, not another ML model.


In [16]:
def recommend_after_exam(math_score, reading_score, writing_score):
    """Recommend a subject to review using actual scores from 0 to 100."""
    scores = {
        "Mathematics": math_score,
        "Reading": reading_score,
        "Writing": writing_score,
    }

    for subject, score in scores.items():
        if not isinstance(score, (int, float, np.integer, np.floating)):
            raise TypeError(f"{subject} score must be numeric.")
        if not np.isfinite(score) or not 0 <= score <= 100:
            raise ValueError(f"{subject} score must be between 0 and 100.")

    lowest_score = min(scores.values())
    weakest_subjects = [
        subject for subject, score in scores.items()
        if score == lowest_score
    ]

    if len(weakest_subjects) == 1:
        subject_text = weakest_subjects[0]
        recommendation = (
            f"Prioritize {subject_text}. Review foundational concepts, work through "
            "guided examples, and practise related questions."
        )
    else:
        subject_text = ", ".join(weakest_subjects)
        recommendation = (
            f"Your lowest scores are tied in {subject_text}. Review the core concepts "
            "in these subjects and practise related questions."
        )

    return {
        "scores": scores,
        "lowest_score": lowest_score,
        "weakest_subjects": weakest_subjects,
        "recommendation": recommendation,
    }

### 3.2 Try a Sample Post-exam Recommendation

The example below uses illustrative scores. Change the values to see how the rule identifies the lowest-scoring subject.


In [17]:
post_exam_result = recommend_after_exam(
    math_score=55,
    reading_score=78,
    writing_score=72,
)

print("Exam scores:")
display(pd.DataFrame(
    list(post_exam_result["scores"].items()),
    columns=["Subject", "Score"],
))
print("Lowest score:", post_exam_result["lowest_score"])
print("Subject(s) to review:", ", ".join(post_exam_result["weakest_subjects"]))
print("Recommendation:", post_exam_result["recommendation"])

Exam scores:


,Subject,Score
0,Mathematics,55
1,Reading,78
2,Writing,72


Lowest score: 55
Subject(s) to review: Mathematics
Recommendation: Prioritize Mathematics. Review foundational concepts, work through guided examples, and practise related questions.


## Section 4 — Verification and Viva Notes

We finish by checking that inference behaves as expected and summarizing the project's important limitations.


### 4.1 Run Verification Checks

These checks confirm that the saved artifacts produce a valid mastery category and probability distribution, and that the post-exam rule identifies the expected subject in the example.


In [18]:
probability_total = sample_result["probabilities"]["Probability"].sum()
assert sample_result["predicted_mastery"] in CLASS_NAMES
assert np.isclose(probability_total, 1.0, atol=1e-6)
assert post_exam_result["weakest_subjects"] == ["Mathematics"]

print("All inference and recommendation checks passed.")
print(f"Probability total: {probability_total:.6f}")

All inference and recommendation checks passed.
Probability total: 1.000000


### 4.2 Summary and Limitations

**What we implemented**
- Loaded the trained, unweighted Logistic Regression model and its fitted preprocessor.
- Predicted a mastery category from five categorical profile features.
- Displayed model class probabilities and mapped the predicted class to a practice suggestion.
- Implemented a separate rule-based recommendation using actual subject scores.
- Added verification checks for the loaded artifacts and example outputs.

**Limitations to explain in a viva**
- The mastery categories are engineered from average exam scores using our chosen thresholds.
- The input variables are limited observational features; predictions should not be treated as causal judgments about students.
- The baseline model had weak recall for the Low class in the held-out test evaluation. This is an educational prototype, not a reliable high-stakes assessment tool.
- The dataset has no question-level difficulty labels or topic-level scores, so the system cannot predict individual question difficulty or diagnose specific topics.
- The recommendation mapping is a simple rule, not a learned recommendation algorithm.


### 4.3 Viva Questions

**Q1. Why do we load the saved preprocessor?**  
To ensure new inputs are encoded using the same fitted categories and feature layout used during model training.

**Q2. What is the difference between the two recommendation modes?**  
The pre-exam mode uses the classifier's predicted mastery category. The post-exam mode uses actual subject scores and a simple rule to identify the weakest subject.

**Q3. Does the system predict the difficulty of individual questions?**  
No. The dataset has no question-level labels. It predicts an engineered student mastery category and maps that category to a broad practice suggestion.
